# Processing Predicates

This notebook demonstrates how to load positive and negative predicates from pickle files, build a domain, and initialize a `LogicModel`.

**Strategy:**
To handle a large number of predicates without exploding memory usage (due to the dense tensor implementation), we employ a "Dense Core" strategy:
1.  **Filter Pronouns:** Remove predicates involving generic pronouns (he, she, it, etc.).
2.  **Identify Core Elements:** Find the Top 300 most frequent elements in the remaining data.
3.  **Filter Predicates:** Keep only predicates where *all* involved elements are in this Top 300 set.

This allows us to process thousands of predicates while keeping the domain size manageable (~300).

In [1]:
import pickle
import LogicModel as m
import numpy as np
import collections

## 1. Load and Filter Data

In [2]:
# Load ALL data
with open('Predicates/FILTERED-predicates.pickle', 'rb') as f:
    raw_pos = pickle.load(f)

with open('Predicates/FILTERED-negative_predicates.pickle', 'rb') as f:
    raw_neg = pickle.load(f)

print(f"Raw Positive: {len(raw_pos)}")
print(f"Raw Negative: {len(raw_neg)}")

# 1. Filter Pronouns
pronouns = {'he', 'she', 'it', 'him', 'her', 'they', 'them', 'we', 'us', 'you',
            'He', 'She', 'It', 'Him', 'Her', 'They', 'Them', 'We', 'Us', 'You',
            'this', 'This', 'that', 'That', 'these', 'These', 'those', 'Those', 
            'I', 'me', 'Me', 'my', 'My', 'myself', 'Myself'}

def filter_pronouns(pred_list):
    cleaned = []
    for s, p, o in pred_list:
        if s in pronouns: continue
        if o is not None and o in pronouns: continue
        cleaned.append((s, p, o))
    return cleaned

clean_pos = filter_pronouns(raw_pos)
clean_neg = filter_pronouns(raw_neg)

# 2. Identify Top Elements (The "Dense Core")
element_counts = collections.Counter()

for s, p, o in clean_pos + clean_neg:
    element_counts[s] += 1
    if o: element_counts[o] += 1

TOP_N = 300
top_elements = set([e for e, c in element_counts.most_common(TOP_N)])

print(f"Selected Top {TOP_N} frequent elements.")

# 3. Final Filter
def keep_core(pred_list):
    final = []
    for s, p, o in pred_list:
        if s not in top_elements: continue
        if o is not None and o not in top_elements: continue
        final.append((s, p, o))
    return final

pos_predicates = keep_core(clean_pos)
neg_predicates = keep_core(clean_neg)

print(f"Final Positive Predicates: {len(pos_predicates)}")
print(f"Final Negative Predicates: {len(neg_predicates)}")

Raw Positive: 30248
Raw Negative: 30145
Selected Top 300 frequent elements.
Final Positive Predicates: 4039
Final Negative Predicates: 5611


## 2. Process Data for LogicModel

We now organize these filtered predicates into the structures required by `LogicModel`.

In [3]:
domain_set = set()
unary_preds_dict = {}
binary_preds_dict = {}

def add_to_domain(elem):
    if elem is not None:
        domain_set.add(elem)

# Process Positive Predicates (True)
for subj, pred, obj in pos_predicates:
    add_to_domain(subj)
    add_to_domain(obj)
    
    if obj is None:
        # Unary
        if pred not in unary_preds_dict:
            unary_preds_dict[pred] = []
        # Add as simple element (implies prob=1.0)
        unary_preds_dict[pred].append(subj)
    else:
        # Binary
        if pred not in binary_preds_dict:
            binary_preds_dict[pred] = []
        binary_preds_dict[pred].append((subj, obj))

# Process Negative Predicates (False)
for subj, pred, obj in neg_predicates:
    add_to_domain(subj)
    add_to_domain(obj)
    
    if obj is None:
        # Unary
        if pred not in unary_preds_dict:
            unary_preds_dict[pred] = []
        # Add as tuple with prob=0.0
        unary_preds_dict[pred].append((subj, 0.0))
    else:
        # Binary
        # Important: Ensure the predicate exists in the dictionary even if the list is empty.
        if pred not in binary_preds_dict:
            binary_preds_dict[pred] = []
        # We do NOT add the pair to the list, so it defaults to False.

domain_list = sorted(list(domain_set))
print(f"Domain Size: {len(domain_list)}")
print(f"Number of Unary Predicates: {len(unary_preds_dict)}")
print(f"Number of Binary Predicates: {len(binary_preds_dict)}")

Domain Size: 300
Number of Unary Predicates: 2475
Number of Binary Predicates: 465


## 3. Build Logic Model

In [4]:
model = m.LogicModel(
    listOfElements=domain_list,
    dictionaryOfUnaryPredicates=unary_preds_dict,
    dictionaryOfBinaryPredicates=binary_preds_dict
)

model.buildAll()

## 4. Verification

We define a helper function `check_truth` to print the result clearly.

In [5]:
def check_unary(pred, elem):
    res = model.unaryOp(pred, elem)
    # res is [[prob_true], [prob_false]]
    is_true = res[0,0] > 0.5
    print(f"'{elem}' is '{pred}'? {is_true} (Scores: True={res[0,0]:.2f}, False={res[1,0]:.2f})")

def check_binary(pred, subj, obj):
    res = model.binaryOp(pred, subj, obj)
    is_true = res[0,0] > 0.5
    print(f"'{subj}' '{pred}' '{obj}'? {is_true} (Scores: True={res[0,0]:.2f}, False={res[1,0]:.2f})")

### Verify Positive Facts
These should return **True**.

In [6]:
# Pick a random positive unary
u_pos_list = [x for x in pos_predicates if x[2] is None]
if u_pos_list:
    u_case = u_pos_list[0]
    check_unary(u_case[1], u_case[0])
else:
    print("No positive unary predicates found in subset.")

# Pick a random positive binary
b_pos_list = [x for x in pos_predicates if x[2] is not None]
if b_pos_list:
    b_case = b_pos_list[0]
    check_binary(b_case[1], b_case[0], b_case[2])
else:
    print("No positive binary predicates found in subset.")

'band' is 'charted'? True (Scores: True=1.00, False=0.00)
'band' 'released' 'album'? True (Scores: True=1.00, False=0.00)


### Verify Negative Facts
These should return **False**.

In [7]:
# Pick a random negative unary
u_neg_list = [x for x in neg_predicates if x[2] is None]
if u_neg_list:
    neg_u_case = u_neg_list[0]
    check_unary(neg_u_case[1], neg_u_case[0])
else:
    print("No negative unary predicates found in subset.")

# Pick a random negative binary
b_neg_list = [x for x in neg_predicates if x[2] is not None]
if b_neg_list:
    neg_b_case = b_neg_list[0]
    check_binary(neg_b_case[1], neg_b_case[0], neg_b_case[2])
else:
    print("No negative binary predicates found in subset.")

'River' is 'has'? False (Scores: True=0.00, False=1.00)
'couple' 'had' 'Egypt'? False (Scores: True=0.00, False=1.00)
